# Phase 2: ML Attrition Model — Step 2.2: Baseline Model

This notebook splits our preprocessed feature matrix into a stratified train/test set (80/20) and trains a baseline Logistic Regression model. It reports Precision, Recall, F1-Score, and ROC-AUC, and discusses why accuracy is not a suitable metric for this imbalanced classification task.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

proc_dir = os.path.join("data", "processed")

## 1. Load Feature Matrix

In [2]:
df_fm = pd.read_csv(os.path.join(proc_dir, "feature_matrix.csv"))
print(f"Feature Matrix Shape: {df_fm.shape}")

y = df_fm["Attrition"]
X = df_fm.drop(columns=["Attrition"])

print(f"Features Shape: {X.shape}")
print(f"Target Shape: {y.shape}")
print(f"Class Balance:\n{y.value_counts(normalize=True)}")

Feature Matrix Shape: (1470, 49)
Features Shape: (1470, 48)
Target Shape: (1470,)
Class Balance:
Attrition
0    0.838776
1    0.161224
Name: proportion, dtype: float64


## 2. Stratified Train-Test Split (80/20)
We perform a stratified split using `random_state=42` to preserve class proportions in both training and test subsets.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train shapes: X={X_train.shape}, y={y_train.shape}")
print(f"Test shapes: X={X_test.shape}, y={y_test.shape}")
print(f"Train class balance:\n{y_train.value_counts(normalize=True)}")
print(f"Test class balance:\n{y_test.value_counts(normalize=True)}")

Train shapes: X=(1176, 48), y=(1176,)
Test shapes: X=(294, 48), y=(294,)
Train class balance:
Attrition
0    0.838435
1    0.161565
Name: proportion, dtype: float64
Test class balance:
Attrition
0    0.840136
1    0.159864
Name: proportion, dtype: float64


## 3. Train Baseline Logistic Regression Model
We train a standard `LogisticRegression` model with `random_state=42` and `max_iter=1000` to guarantee convergence.

In [4]:
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_train, y_train)
print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## 4. Evaluate Baseline Performance
We predict on the test set and calculate classification metrics.

In [5]:
y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("=== Baseline Model Metrics ===")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred))

=== Baseline Model Metrics ===
Accuracy:  0.8673
Precision: 0.6538
Recall:    0.3617
F1-Score:  0.4658
ROC-AUC:   0.8103

=== Classification Report ===
              precision    recall  f1-score   support

           0       0.89      0.96      0.92       247
           1       0.65      0.36      0.47        47

    accuracy                           0.87       294
   macro avg       0.77      0.66      0.70       294
weighted avg       0.85      0.87      0.85       294


=== Confusion Matrix ===
[[238   9]
 [ 30  17]]


## Why Accuracy is Not a Reliable Metric (Class Imbalance)
In our dataset, the class balance is highly skewed: approximately **83.9% of employees stay (No)** and only **16.1% leave (Yes)**.

If we built a naive, dummy model that predicted "No" for every single employee, that model would achieve a **83.9% accuracy** while failing to identify a single actual attrition event. In other words, accuracy makes a model look highly successful even when it has zero utility for the business problem.

To solve attrition, the company needs to identify employees who are at risk of leaving so it can intervene. Therefore:
- **Recall** (Sensitivity) is critical because it tells us the percentage of actual attrition cases the model successfully identified. A false negative (missing someone who will leave) is highly costly.
- **Precision** is important because it represents the accuracy of our positive predictions (preventing "alarm fatigue" from wasting retention budgets on employees who had no intention of leaving).
- **F1-Score** provides a balanced harmonic mean of precision and recall.
- **ROC-AUC** measures the model's ability to rank risk probabilities correctly regardless of the classification threshold.